In [3]:
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json
from dotenv import load_dotenv


from pathlib import Path

from utils.mistral_create_datasets import train_data

# Preliminari

In [4]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.OPENAI_GPT_5_2
RESULTS_FILE_NAME = 'results_' + 'gpt-5.2' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [5]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [6]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 65
len(validation_data) = 66


# Generazione

## Preliminari generazione

In [8]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=output_format
    )
    return response

In [9]:
# Esempio
if True:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [15]:
result.output_parsed.model_dump(mode='json')

{'morfologia': 'solido_polipoide',
 'ore_inizio': None,
 'ore_fine': None,
 'spessore_parietale': None,
 'estensione_cranio_caudale': None,
 'distanza_oai': None,
 'posizione': {'basso': 'si', 'medio': 'no', 'alto': 'no', 'giunzione': 'no'},
 'riflessione_peritoneale_anteriore': 'non_valutabile',
 'infiltrazione_tessuto_adiposo': 'no',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'si',
 'infiltrazione_organi_dettagli': {'pavimento_pelvico': 'si', 'altro': 'no'},
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'no',
 'numero_linfonodi_non_conosciuto': 'non_conosciuto',
 'linfonodi_sospetti': 0,
 'sedi_linfonodi': {'mesorettali': 'no',
  'rettali_superiori': 'no',
  'otturatori': 'si',
  'iliaci': 'no',
  'altro': 'si'},
 'depositi_tumorali': 'si',
 'emvi': '+',
 'stadio_T': 'T4b',
 'stadio_N': 'N+',
 'stadio_N1c': 'si',
 'mrf': '-',
 'metastasi': 'MX'}

In [11]:
if False: # To see the full output
    pprint(result.model_dump())
if False:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

{'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1771071613.0,
 'conversation': None,
 'created_at': 1771071610.0,
 'error': None,
 'frequency_penalty': 0.0,
 'id': 'resp_0680dec2f15d5c85006990687a08148191bfbecc94a29a6ef2',
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt-5.2-2025-12-11',
 'object': 'response',
 'output': [{'content': [{'annotations': [],
                          'logprobs': [],
                          'parsed': {'coinvolgimento_fascia_mesorettale': 'no',
                                     'coinvolgimento_riflessione_peritoneale': 'no',
                                     'depositi_tumorali': 'si',
                                     'distanza_oai': None,
                                     'emvi': '+',
                                     'estensione_cranio_caudale': None,
                                     'infiltrazione_organi_dettagli': {'altr

C:\Users\lucat\PycharmProjects\PRIN\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(PydanticSerializationUnexpectedValue: Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RectalCancerStagingData(m...mrf='-', metastasi='MX'), input_type=RectalCancerStagingData]
PydanticSerializationUnexpectedValue: Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...rf='-', metastasi='MX')), input_type=ParsedResponseOutputText[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ParsedResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...pleted', type='message'), input_type=ParsedResponseOutputMessage[~TextFormatT]])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - serialized value 

## Inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [16]:
print(MODEL)
df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    actual.append(real.model_dump(mode='json'))
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.output_parsed.model_dump(mode='json'))

gpt-5.2-2025-12-11


100%|██████████| 131/131 [10:32<00:00,  4.83s/it]


In [17]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [18]:
output.model_dump()

{'id': 'resp_0a65d0a86c6d21120069906c04d41c819fb699804f988cc8ec',
 'created_at': 1771072518.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5.2-2025-12-11',
 'object': 'response',
 'output': [{'id': 'msg_0a65d0a86c6d21120069906c0735d0819facd2bce8b0072e67',
   'content': [{'annotations': [],
     'text': '{"morfologia":"solido_anulare","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":40,"distanza_oai":50,"posizione":{"basso":"no","medio":"si","alto":"si","giunzione":"no"},"riflessione_peritoneale_anteriore":"non_valutabile","infiltrazione_tessuto_adiposo":"si_5mm_plus","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":"no","altro":"no"},"coinvolgimento_riflessione_peritoneale":"si","coinvolgimento_fascia_mesorettale":"no","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":2,"sedi_linfonodi":{"mesorettali":"si","